# Feature Engineering

Create engineered features using `src.feature_engineer.FeatureEngineer` and save the result.

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))
from src.feature_engineer import FeatureEngineer
from src.data_loader import DataLoader
from src.preprocessing import FraudDataPreprocessor
import pandas as pd

In [2]:
# Load processed fraud data if available, otherwise run preprocessing
processed_path = '../data/processed/fraud_data_processed.csv'
try:
    df = pd.read_csv(processed_path)
    print('Loaded processed data:', processed_path)
except Exception as e:
    print('Processed file not found, loading raw and preprocessing...')
    dl = DataLoader(data_dir='../data/raw')
    fraud_df = dl.load_fraud_data('Fraud_Data.csv')
    ip_df = dl.load_ip_country_data('IpAddress_to_Country.csv')
    prep = FraudDataPreprocessor()
    fraud_clean = prep.clean_data(fraud_df)
    fraud_features = prep.create_time_features(fraud_clean)
    df = prep.merge_with_ip_data(fraud_features, ip_df)
    df.to_csv(processed_path, index=False)
    print('Saved processed data to', processed_path)

Loaded processed data: ../data/processed/fraud_data_processed.csv


In [3]:
# Run feature engineering
fe = FeatureEngineer()
df_ts = df.copy()
# Ensure purchase_time is datetime
if 'purchase_time' in df_ts.columns:
    df_ts['purchase_time'] = pd.to_datetime(df_ts['purchase_time'])
df_features = fe.create_all_features(df_ts)
print('Engineered features shape:', df_features.shape)
df_features.head()

Created 7 new features
Engineered features shape: (151112, 24)


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,...,purchase_month,time_since_signup,same_day_purchase,purchase_value_log,hour_of_day,day_of_week,month,source_encoded,browser_encoded,sex_encoded
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,...,4,1251.856111,0,3.555348,2,5,4,2,0,1
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,...,6,4.984444,0,2.833213,1,0,6,0,0,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,...,1,0.000278,1,2.772589,18,3,1,2,3,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,...,5,136.690278,0,3.806662,13,0,5,2,4,1
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,...,9,1211.516944,0,3.688879,18,2,9,0,4,1


In [4]:
# Save engineered features
out_path = '../data/processed/fraud_data_features.csv'
df_features.to_csv(out_path, index=False)
print('Saved engineered features to', out_path)

Saved engineered features to ../data/processed/fraud_data_features.csv
